## Отбор признаков

In [58]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('ggplot')
from sklearn import feature_selection
from sklearn import model_selection
from scipy import stats
from sklearn import metrics
from sklearn.feature_selection import RFE
from sklearn import linear_model

In [59]:
def get_metrics(X_train:pd.DataFrame, y_train:pd.Series, X_test:pd.DataFrame, y_test:pd.Series) -> float:
    """ Вычисление метрик MAE, MAPE, MSE, RMSE, R2-score

    Args:
        X_train (pd.DataFrame): тренировочный датасет
        y_train (pd.Series): тренировочный целевой вектор
        X_test (pd.DataFrame): тестовый датасет
        y_test (pd.Series): тестовый целевой вектор

    Returns:
        float: метрики MAE, MAPE, MSE, RMSE, R2-score
    """
    # присваиваем модели тип - линейную регрессию, обучаем её, предсказываем значения на тестовой выборке
    model = linear_model.LinearRegression()
    model.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)

    # вычисление метрик, округление до 3 знака после запятой и перевод к типу float
    mae = float(np.round(metrics.mean_absolute_error(y_test, y_test_pred), 3))
    mape = float(np.round(metrics.mean_absolute_percentage_error(y_test, y_test_pred), 3))
    mse = float(np.round(metrics.mean_squared_error(y_test, y_test_pred), 3))
    rmse = float(np.round(np.sqrt(metrics.mean_squared_error(y_test, y_test_pred)), 3))
    r2 = float(np.round(metrics.r2_score(y_test, y_test_pred), 3))

    # возвращаем кортеж с метриками
    return mae, mape, mse, rmse, r2

In [60]:
# читаем исходный датасет
data = pd.read_excel('./data/data_ford_price.xlsx')
data.head()

,price,year,condition,cylinders,odometer,title_status,transmission,drive,size,lat,long,weather
0,43900,2016,4,6,43500,clean,automatic,4wd,full-size,36.471500,-82.483400,59.0
1,15490,2009,2,8,98131,clean,automatic,4wd,full-size,40.468826,-74.281734,52.0
2,2495,2002,2,8,201803,clean,automatic,4wd,full-size,42.477134,-82.949564,45.0
3,1300,2000,1,8,170305,rebuilt,automatic,4wd,full-size,40.764373,-82.349503,49.0
4,13865,2010,3,8,166062,clean,automatic,4wd,NaN,49.210949,-123.114720,NaN


In [ ]:
# удаляем пропуски
data.dropna(inplace = True)

In [ ]:
# преобразовывем признак condition
data['condition'] = data['condition'].astype('object')

In [63]:
# оставляем только числовые признаки
num_cols = [col for col in data.columns if data[col].dtypes != 'object']
data_num = data[num_cols].copy()

# выделяем целевой признак
y_num = data_num['price']
X_num = data_num.drop(columns='price')

# разделяем данные на тренировочную и тестовую выборки
X_train_num, X_test_num, y_train_num, y_test_num = model_selection.train_test_split(
    X_num, y_num, 
    test_size=0.2, 
    random_state=42
)

In [64]:
num_metrics = get_metrics(X_train_num, y_train_num, X_test_num, y_test_num)
print(f'MAE score: {num_metrics[0]}\nR2 score: {num_metrics[4]}')

MAE score: 4620.233
R2 score: 0.634


In [ ]:
# выбираем лучшие признаки по методу рекурсивного исключения признаков (RFE)
estimator = linear_model.LinearRegression()
selector_rfe = feature_selection.RFE(estimator, n_features_to_select=3, step=1)
selector_rfe.fit(X_train_num, y_train_num)

# выводим отобранные признаки
rfe_cols = list(selector_rfe.get_feature_names_out())
rfe_cols

['year', 'cylinders', 'lat']

In [ ]:
# выбираем лучшие признаки по методу "выбор k лучших переменных" (SelectKBest)
selector_skb = feature_selection.SelectKBest(feature_selection.f_regression, k=3)
selector_skb.fit(X_train_num, y_train_num)

# выводим отобранные признаки
skb_cols = list(selector_skb.get_feature_names_out())
skb_cols

['year', 'cylinders', 'odometer']

In [ ]:
# обученение регрессии на RFE-признаках
X_train_rfe = X_train_num[rfe_cols]
X_test_rfe = X_test_num[rfe_cols]

# выводим метрики
fre_metrics = get_metrics(X_train_rfe, y_train_num, X_test_rfe, y_test_num)
print(f'MAE score: {fre_metrics[0]}\nR2 score: {fre_metrics[4]}')

MAE score: 5107.525
R2 score: 0.573


In [ ]:
# обученение регрессии на выбранных k лучших столбцах
X_train_skb = X_train_num[skb_cols]
X_test_skb = X_test_num[skb_cols]

# выводим метрики
skb_metrics = get_metrics(X_train_skb, y_train_num, X_test_skb, y_test_num)
print(f'MAE score: {skb_metrics[0]}\nR2 score: {skb_metrics[4]}')

MAE score: 4627.369
R2 score: 0.631


## Выводы

- Средняя абсолютная ошибка (MAE - Mean Absolute Error) — это метрика для оценки регрессионных моделей, которая показывает среднюю величину абсолютных ошибок между предсказанными и фактическими значениями. MAE показывает, "в среднем на сколько ошибается модель" в единицах измерения целевой переменной.
- R² (R-квадрат, коэффициент детерминации) — это ключевая метрика качества регрессионных моделей, которая показывает долю дисперсии целевой переменной, объясненную моделью. Показывает, насколько хорошо модель объясняет вариацию данных относительно простого среднего значения.

Для 3-х моделей у нас получились следующие результаты:
1) модель линейной регрессии для всех числовых признаков: MAE score: 4620.233, R2 score: 0.634
2) модель линейной регрессии для признаков ['year', 'cylinders', 'lat'], полученных методом рекурсивного исключения признаков (RFE): MAE score: 5107.525, R2 score: 0.573
3) модель линейной регрессии для признаков ['year', 'cylinders', 'odometer'], полученных методом выбор k лучших признаков (SelectKBest): MAE score: 4627.369, R2 score: 0.631

Если сравнивать показатели регрессий на 3-х признаках, то лидирует модель №3
Если сравнивать первую и третью модели, то стоит отдать предпочтение регрессии для всех признаков, так как она более детализированна. По показателям различия незначительны.